In [ ]:
# ======================================================
# Notebook: Hyperparameter optimisation (Logistic Regression)
# Inputs: (30,4) | Output: (30,)
# Goal: minimise difference from baseline (maximize negative)
# ======================================================

import numpy as np
from sklearn.linear_model import LogisticRegression

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (30,4)
y_raw = np.load("/mnt/data/initial_outputs.npy") # (30,)

# Transform objective (minimise -> maximise)
y = -y_raw

# Convert to binary (best-performing region)
threshold = np.percentile(y, 75)
y_bin = (y >= threshold).astype(int)

# Train logistic regression surrogate
model = LogisticRegression(max_iter=1000)
model.fit(X, y_bin)

# Generate candidate samples within bounds
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(4)]
num_candidates = 5000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], num_candidates) for b in bounds
])

# Predict probability of optimal region
probs = model.predict_proba(X_grid)[:,1]

# Select next (10,4) hyperparameter sets
top_idx = np.argsort(probs)[-10:]
next_points = X_grid[top_idx]

print("Next (10,4) hyperparameter candidates:")
print(next_points)